In [ ]:
from selenium import webdriver
from time import sleep
import random
from bs4 import BeautifulSoup as bs
from selenium.common.exceptions import NoSuchElementException

from selenium.webdriver.common.by import By

In [2]:
url = 'https://pe.computrabajo.com/'

In [4]:
htmls = list()

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementNotInteractableException
from time import sleep
import random

terminos = [
    'desarrollador de software',
    'analista de datos',
    'cientifico de datos',
    'ingeniero de datos',
    'devops',
    'ciberseguridad',
]

htmls = []

driver = webdriver.Chrome()
driver.maximize_window()
driver.set_page_load_timeout(60)
driver.implicitly_wait(10)  
try:
    for termino in terminos:
        driver.get(url)
        sleep(2)

        try:
            driver.find_element(By.XPATH, '/html/body/div/div/a').click()
            sleep(1)
        except (NoSuchElementException, ElementNotInteractableException):
            pass

        search_box = driver.find_element(By.ID, 'prof-cat-search-input')
        search_box.clear()
        search_box.send_keys(termino)
        sleep(1)
        driver.find_element(By.ID, 'search-button').click()
        sleep(2)

        try:
            driver.find_element(By.XPATH, '//*[@id="pop-up-webpush-sub"]/div[2]/div/button[1]').click()
            sleep(1)
        except (NoSuchElementException, ElementNotInteractableException):
            pass

        # Paginación
        while True:
            sleep(random.uniform(1.5, 2.5))
            htmls.append(driver.page_source)

            try:
                next_button = driver.find_element(By.CSS_SELECTOR, 'span.w48:nth-child(2)')
                if 'disabled' in next_button.get_attribute('class').split():
                    break
                next_button.click()
            except NoSuchElementException:
                break
finally:
    driver.quit()

In [ ]:
from bs4 import BeautifulSoup as bs

all_links = []

for html in htmls:
    soup = bs(html, 'html.parser')
    links = soup.find_all('a', {'class': 'js-o-link fc_base'})
    all_links.extend([link.get('href') for link in links if link.get('href')])

all_links = list(dict.fromkeys(all_links))

print(f"Total de links extraídos: {len(all_links)}")

Total de links extraídos: 1794


In [13]:
import json
import random
from time import sleep
from datetime import datetime, timezone
from urllib.parse import urljoin
from curl_cffi import requests as rq
from bs4 import BeautifulSoup as bs

BASE_URL = "https://pe.computrabajo.com"

def extraer_oferta(html, url_origen, fecha_consulta, fase="retrospectiva"):
    soup = bs(html, "html.parser")

    def obtener_meta(prop):
        tag = soup.find("meta", {"itemprop": prop})
        return tag["content"].strip() if tag and tag.get("content") else None

    bloque_oferta = soup.find("div", {"div-link": "oferta"})
    tags = [t.get_text(strip=True) for t in bloque_oferta.find_all("span", class_="tag base")] if bloque_oferta else []

    return {
        "id": obtener_meta("value"),
        "titulo": obtener_meta("title"),
        "empresa": obtener_meta("name"),
        "ubicacion": f"{obtener_meta('addressRegion')}, {obtener_meta('addressLocality')}",
        "industria": obtener_meta("industry"),
        "fecha_publicacion": obtener_meta("datePosted"),
        "fecha_expiracion": obtener_meta("validThrough"),
        "tipo_empleo": obtener_meta("employmentType"),
        "tags_oferta": tags,  # salario declarado, modalidad, contrato
        "descripcion": obtener_meta("description"),
        "url": obtener_meta("url") or url_origen,
        "first_seen": fecha_consulta,
        "last_seen": fecha_consulta,
        "fase": fase,
    }

ofertas = []
fecha_consulta = datetime.now(timezone.utc).isoformat()

for link in all_links:
    url_completa = urljoin(BASE_URL, link)
    try:
        resp = rq.get(url_completa, impersonate="chrome", timeout=15)
        if resp.status_code == 200:
            html = resp.content.decode("utf-8", errors="replace")
            ofertas.append(extraer_oferta(html, url_completa, fecha_consulta))
        else:
            print(f"Error {resp.status_code} en {url_completa}")
    except Exception as e:
        print(f"Fallo en {url_completa}: {e}")

    sleep(random.uniform(0.5, 1.5))

with open("ofertas_computrabajo.json", "w", encoding="utf-8") as f:
    json.dump(ofertas, f, ensure_ascii=False, indent=2)

print(f"Guardadas {len(ofertas)} ofertas en ofertas_computrabajo.json")

Fallo en https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-analista-de-sistemas-bigdatapythonc-en-puente-piedra-32463C38725E69B261373E686DCF3405#lc=ListOffers-Score3-17: Failed to perform, curl: (6) Could not resolve host: pe.computrabajo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
Fallo en https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-analista-de-planeamiento-planificacion-consecionario-de-autos-en-lima-95CD33D7F85B4EC661373E686DCF3405#lc=ListOffers-Score3-8: Failed to perform, curl: (6) Could not resolve host: pe.computrabajo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
Fallo en https://pe.computrabajo.com/ofertas-de-trabajo/oferta-de-trabajo-de-analista-de-planeamiento-estrategico-sede-san-borja-lima-en-san-borja-2ED6ACAE09DA45D461373E686DCF3405#lc=ListOffers-Score3-9: Failed to perform, curl: (6) Could not resolve host: pe.computrabajo.com. See https://curl.se/libcurl/c/libc